<a href="https://colab.research.google.com/github/prpNn0p/machinetranslation/blob/main/simpletransformers_mt_th_en.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Original Note


In [ ]:
'''
How to Train an mT5 Model for Translation With Simple Transformers
The mT5 model is pre-trained on over a hundred different languages. 
Let’s see how we can leverage this to train a bilingual translation model 
for a low-resource language — Sinhalese.
'''






'''
mT5 is a multilingual Transformer model pre-trained on a dataset (mC4) containing text from 101 different languages. 
The architecture of the mT5 model (based on T5) is designed to support any Natural Language Processing task (classification, NER, question answering, etc.) by reframing the required task as a sequence-to-sequence task.

In other words — text goes in, and text comes out. For example, in a classification task, the input to the model can be the text sequence to be classified, 
and the output from the model will be the class label for the sequence. 
For translation, this is even more straight forward. The text that goes in is in one language, and the text that comes out is in another.

Considering the multilingual capabilities of mT5 and the suitability of 
the sequence-to-sequence format for language translation, let’s see how we can fine-tune an mT5 model for machine translation. 
For this article, we’ll be training a translation model to translate between Sinhalese (my native language!) and English. 
It’s quite challenging to train good translation models for low-resource languages like Sinhalese because of, well, the low availability of resources (training data). 

Hopefully, the multilingual pre-training on a huge dataset (which includes Sinhalese, 
although not a lot of it) will help the mT5 model compensate for inadequate training data in the form of direct Sinhalese-English (and vice versa) sequences.

We’ll be using the Simple Transformers library (built on the Huggingface Transformers library) to train the mT5 model. 
The training and testing data will be obtained from The Tatoeba Translation Challenge. 
The graphs and charts are generated from Weights & Biases, which is supported natively in Simple Transformers for experiment tracking and hyperparameter optimization.

Note: You can find all the code in this article in the examples/t5/mt5_translation directory (link) of the Simple Transformers Github repo.
'''


'''
Outline

    Installing Simple Transformers
    Downloading a dataset for translation
    Training the model
    Evaluating the model — Calculating the BLEU score
    Wrap up

'''

##Setup

You can find the most up-to-date installation instructions in the Simple Transformers documentation.

1. Install Anaconda or Miniconda Package Manager from here.

2. Create a new virtual environment and install packages.



In [ ]:
# conda create -n st python pandas tqdm sacrebleu
# conda activate st

3. If using CUDA:

In [ ]:
# conda install pytorch>=1.6 cudatoolkit=10.2 -c pytorch

else:

In [ ]:
# conda install pytorch cpuonly -c pytorch

4. Install simple transformers.

In [ ]:
!pip install simpletransformers

     |████████████████████████████████| 248 kB 14.1 MB/s 
     |████████████████████████████████| 3.8 MB 85.7 MB/s 
     |████████████████████████████████| 1.7 MB 58.6 MB/s 
     |████████████████████████████████| 9.9 MB 53.0 MB/s 
     |████████████████████████████████| 6.5 MB 42.9 MB/s 
     |████████████████████████████████| 1.2 MB 54.3 MB/s 
     |████████████████████████████████| 43 kB 1.9 MB/s 
     |████████████████████████████████| 312 kB 53.3 MB/s 
     |████████████████████████████████| 596 kB 70.6 MB/s 
     |████████████████████████████████| 895 kB 55.7 MB/s 
     |████████████████████████████████| 67 kB 5.7 MB/s 
     |████████████████████████████████| 181 kB 56.8 MB/s 
     |████████████████████████████████| 144 kB 55.3 MB/s 
     |████████████████████████████████| 63 kB 1.6 MB/s 
     |████████████████████████████████| 212 kB 74.1 MB/s 
     |████████████████████████████████| 134 kB 64.5 MB/s 
     |████████████████████████████████| 1.1 MB 68.7 MB/s 
     |██████████████

## Data Preparation

The training and test can be obtained from the Tatoeba Translation Challenge data page. You’ll also find datasets for a whole host of other languages there (including Sindarin, a language spoken by Elves in The Lord of the Rings 😀).

If you want to try training a translation model for another language, you can download the dataset for that language instead of Sinhalese. All the other steps in this article apply to any language dataset.

If you are too lazy to search for the dataset, here’s the direct link. 😜

1. Download the (compressed) translation dataset and extract the archive.
2. Extract the train.trg.gz and train.src.gz archives (yes, the training data is in archives inside archives).
3. Check whether you have thetrain.trg, train.src, test.trg, test.src files in the data/eng-sin/ directory. (If not, it’s probably easiest to move the files to this location as the code examples will assume that these files can be found here)

Now, let’s build the tsv files that we will use to train and test our mT5 model.

In [ ]:
pip install -U --no-cache-dir gdown --pre

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
    Preparing wheel metadata ... done
  Created wheel for gdown: filename=gdown-4.4.0-py3-none-any.whl size=14774 sha256=413ecc676e4d5b9286d6d93e567611a1063e9c700df94690fbeeaf66e3f012e8
  Stored in directory: /tmp/pip-ephem-wheel-cache-zrsi_xf_/wheels/fb/c3/0e/c4d8ff8bfcb0461afff199471449f642179b74968c15b7a69c
Successfully built gdown
  Attempting uninstall: gdown
    Found existing installation: gdown 4.2.2
    Uninstalling gdown-4.2.2:
      Successfully uninstalled gdown-4.2.2


In [ ]:
!gdown --id 1T6CtK1x00ffMfNvAzpTL6Ka0_3Wuevhm

/usr/local/lib/python3.7/dist-packages/gdown/cli.py:131: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  category=FutureWarning,
Downloading...
From: https://drive.google.com/uc?id=1T6CtK1x00ffMfNvAzpTL6Ka0_3Wuevhm
To: /content/scb-mt-en-th-2020.zip
100% 106M/106M [00:00<00:00, 152MB/s] 


In [ ]:
# !gdown --id 1m0PRbjnn1xGE0SoE4REhi6R9NQ2bkvap

In [ ]:
!gdown --id 18J1a6YEIUFWUHtW9T6uS0h7gBJOZqxOS

/usr/local/lib/python3.7/dist-packages/gdown/cli.py:131: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  category=FutureWarning,
Downloading...
From: https://drive.google.com/uc?id=18J1a6YEIUFWUHtW9T6uS0h7gBJOZqxOS
To: /content/Official_TrainningDataset_Machine Translation.zip
100% 238M/238M [00:01<00:00, 159MB/s]


In [ ]:
%mv /content/Official_TrainningDataset_Machine\ Translation.zip /content/Official_Dataset_Machine_Translation.zip

In [ ]:
#!unzip -qq /content/scb-mt-en-th-2020.zip

In [ ]:
!unzip -qq /content/Official_Dataset_Machine_Translation.zip

In [ ]:
import os
import pandas as pd

In [ ]:
# train_df, eval_df = prepare_translation_datasets("data/eng-sin")

In [ ]:
def extract_data(file):
  f = open(file)
  x = f.read()

# # Put data in to the variable predict_x
# predict_x = x.replace("\n", " _ ")
# predict_x = predict_x.replace("\n", " ")
  output = x.split("\n")
  return output



## LST_ENTH680K Datasets

In [ ]:
path_en = '/content/TrainSet/LST_ENTH680K/lowercase/lang.en'
path_th = '/content/TrainSet/LST_ENTH680K/lowercase/lang.th'
lower_case_df = pd.DataFrame(extract_data(path_en), columns = ['en_text'])
lower_case_df['th_text'] = extract_data(path_th)
lower_case_df = lower_case_df[:681539]
lower_case_df

,en_text,th_text
0,thank you but i don't have enough time,ขอบคุณ แต่ ฉัน ไม่ มี เวลา พอ
1,"oh, you know, when i get up i usually do my st...","โอ้ , คุณ รู้ , เมื่อ ฉัน ตื่น ฉัน จะ ออกไป เด..."
2,the duty of a policeman is to safeguard people .,หน้าที่ ของ ตำรวจ คือ การ ปกป้อง ประชาชน
3,it's female vanity.,มัน เป็นความ โอหัง ของ เพศ หญิง .
4,please keep a sharp lookout .,โปรด คอย ดู อย่า เผลอ
...,...,...
681534,"when wijai talked to me on the phone , his voi...",ตอน วิจัย พูด โทรศัพท์ กับ ดิฉัน เสียง เขา แตก...
681535,the bus terminal is on the west side of the ci...,สถานี รถ โดยสาร ทาง ตะวันตก ของ เมือง
681536,encapsulate,ลักษณะ ภาย นอก
681537,"death frightens me , specifically my own death .",ความ ตาย ทำ ให้ ผม กลัว โดย เฉพาะอย่างยิ่ง การ...


# Unused Datasets

## Bible Datasets

In [ ]:
path_en = '/content/TrainSet/OPUS_BiTextCollections/Bible/bible-uedin.en-th.en'
path_th = '/content/TrainSet/OPUS_BiTextCollections/Bible/bible-uedin.en-th.th'
Bible_df = pd.DataFrame(extract_data(path_en), columns = ['en_text'])
Bible_df['th_text'] = extract_data(path_th)
Bible_df

,en_text,th_text
0,In the beginning God created the heavens and t...,ในเริ่มแรกนั้นพระเจ้าทรงเนรมิตสร้างฟ้าและแผ่นด...
1,Now the earth was formless and empty. Darkness...,แผ่นดินโลกนั้นก็ปราศจากรูปร่างและว่างเปล่าอยู่...
2,"God said, ""Let there be light,"" and there was ...",วัน ที่ หนึ่ง ปรากฏ มีค วาม สว่าง เกิด ขึ้น พร...
3,"God saw the light, and saw that it was good. G...",พระเจ้า ทรง เห็น ว่าความ สว่าง นั้น ดี และ พระ...
4,"God called the light ""day,"" and the darkness h...",พระเจ้า ทรง เรียก ความ สว่าง นั้น ว่า วัน และ ...
...,...,...
124382,For I testify unto every man that heareth the ...,ข้าพเจ้าเป็นพยานแก่ทุกคนที่ได้ยินคำพยากรณ์ในหน...
124383,And if any man shall take away from the words ...,และถ้าผู้ใดตัดข้อความออกจากหนังสือพยากรณ์นี้ พ...
124384,"He which testifieth these things saith, Surely...",คำทรงเตือนและพระสัญญาว่าพระคริสต์จะเสด็จมาพระอ...
124385,The grace of our Lord Jesus Christ be with you...,ขอให้พระคุณแห่งพระเยซูคริสต์องค์พระผู้เป็นเจ้า...


## GNOME Datasets

In [ ]:
path_en = '/content/TrainSet/OPUS_BiTextCollections/GNOME/GNOME.en-th.en'
path_th = '/content/TrainSet/OPUS_BiTextCollections/GNOME/GNOME.en-th.th'
GNOME_df = pd.DataFrame(extract_data(path_en), columns = ['en_text'])
GNOME_df['th_text'] = extract_data(path_th)
GNOME_df

,en_text,th_text
0,Accerciser,Accerciser
1,Give your application an accessibility workout,ช่วยพัฒนาสิ่งอำนวยความสะดวกในโปรแกรมของคุณ
2,Accerciser Accessibility Explorer,Accerciser: เครื่องมือสำรวจสิ่งอำนวยความสะดวก
3,The default plugin layout for the bottom panel,การจัดวางปลั๊กอินสำหรับช่องด้านล่าง
4,The default plugin layout for the top panel,การจัดวางปลั๊กอินสำหรับช่องด้านบน
...,...,...
74,Rows,จำนวนแถว
75,Table Information,ข้อมูลตาราง
76,"name (x,y)","ชื่อ (x,y)"
77,Header:,หัวตาราง:


## JW300 Datasets

In [ ]:
path_en = '/content/TrainSet/OPUS_BiTextCollections/JW300/jw300.en'
path_th = '/content/TrainSet/OPUS_BiTextCollections/JW300/jw300.th'
JW300_df = pd.DataFrame(extract_data(path_en), columns = ['en_text'])
JW300_df['th_text'] = extract_data(path_th)
JW300_df

,en_text,th_text
0,,อนาคต ​ ของ ​ ศาสนา ​ จาก ​ การ ​ พิจารณา ​ ศา...
1,Religion’s Future in View of Its Past,ตอน ​ ที่ 13 : จาก ส .
2,Part 13 ​ — 476 C.E . onward ​ — Out of Darkne...,ศ . 476 เป็น ​ ต้น ​ มา — สิ่ง ​ ที่ ​ ถือ ​ ว...
3,“ Sins committed in the dark are seen in Heave...,“ บาป ​ ซึ่ง ​ ได้ ​ กระทำ ​ ใน ​ ที่ ​ มืด ​ ...
4,IN APRIL 1988 the Church in the Soviet Union r...,เดือน ​ เมษายน 1988 คริสต์ ​ จักร ​ ใน ​ สหภาพ...
...,...,...
818625,Summarize key points in the material to help y...,คุณ ​ เห็น ​ อะไร ได้ ​ ยิน ​ อะไร และ ​ ได้ ​...
818626,Is there something you can use in your ministr...,ลอง ​ สมมุติ ​ ว่า ​ คุณ ​ เป็น ​ ตัว ​ ละคร ​...
818627,,ใช้ ​ เครื่อง ​ มือ ​ ของ ​ องค์การ ​ เพื่อ ​ ...
818628,,สรุป ​ จุด ​ สำคัญ การ ​ ทำ ​ อย่าง ​ นี้ ​ จะ...


## OpenSubtitles Datasets

In [ ]:
path_en = '/content/TrainSet/OPUS_BiTextCollections/OpenSubtitles/OpenSubtitles.en-th.en'
path_th = '/content/TrainSet/OPUS_BiTextCollections/OpenSubtitles/OpenSubtitles.en-th.th'
opensub_df = pd.DataFrame(extract_data(path_en), columns = ['en_text'])
opensub_df['th_text'] = extract_data(path_th)
opensub_df

,en_text,th_text
0,"Slave in the Magic Mirror, come from the farth...","ทาสในกระจกวิเศษ, มาจากพื้นที่ที่ไกลที่สุด"
1,"Through wind and darkness, I summon thee.",ผ่านลมและความมืดฉันเรียกเจ้า
2,Speak!,พูด!
3,Let me see thy face.,ให้ฉันเห็นพระพักตร์ของ พระองค์
4,"What wouldst thou know, my Queen?",สิ่งที่เจ้าจะรู้ว่าสมเด็จพระราชินี ของฉันได้อย...
...,...,...
3281529,MY LADIES BY SHIN MICHIMA,(เหล่าสุภาพสตรีของผม โดย มิชิมะ ชิน)
3281530,THIS DRAMA IS FICTIONAL.,(ภาพยนตร์ชุดนี้เป็นเรื่องแต่ง)
3281531,ALL CHARACTERS AND ORGANIZATIONS IN IT ARE FIC...,(ตัวละครและองค์กรในเรื่อง ล้วนเป็นเรื่องแต่ง)
3281532,Subtitle translation by Brian Athey,คำบรรยายโดย:


## QED Datasets

In [ ]:
path_en = '/content/TrainSet/OPUS_BiTextCollections/QED/QED.en-th.en'
path_th = '/content/TrainSet/OPUS_BiTextCollections/QED/QED.en-th.th'
qed_df = pd.DataFrame(extract_data(path_en), columns = ['en_text'])
qed_df['th_text'] = extract_data(path_th)
qed_df

,en_text,th_text
0,This is the same problem that we had in the la...,นี่คือปัญหาข้อเดิมที่เราทำในวิดีโอที่แล้ว
1,But instead of trying to figure out whether th...,แต่แทนที่จะหาว่าข้อมูล ที่ได้มาเป็นหลักฐานเพีย...
2,So you could ignore the question right here.,คุณลืมคำถามที่แล้วไปได้
3,You can ignore all of this.,คุณทิ้งทั้งหมดนี่ไป
4,I'm just using that same data to come up with ...,ผมแค่ใช้ข้อมูลเดิมที่มี เพื่อสร้างช่วง ความมั่...
...,...,...
264673,"So anyway, I didn't want to appear defensive, ...",เอาล่ะ ผมไม่อยากปกป้องตัวเอง แต่ผม อยากทำให้ทุ...
264674,Because I don't want to in any way blame those...,เพราะผมอยากให้ใครว่าคนที่คิดว่า บทพิสูจน์เดิมข...
264675,It's my fault because I didn't explain it prop...,ผมผิดเองที่ไม่ได้อธิบายให้ชัด
264676,So hopefully this should provide a little bit ...,งั้นหวังว่านี่คงช่วย ให้เรื่องนี้กระจ่างขึ้น -


## Tanzil Datasets

In [ ]:
path_en = '/content/TrainSet/OPUS_BiTextCollections/Tanzil/Tanzil.en-th.en'
path_th = '/content/TrainSet/OPUS_BiTextCollections/Tanzil/Tanzil.en-th.th'
tanzil_df = pd.DataFrame(extract_data(path_en), columns = ['en_text'])
tanzil_df['th_text'] = extract_data(path_th)
tanzil_df

,en_text,th_text
0,"In the name of Allah, most benevolent, ever-me...",ด้วยพระนามของอัลลอฮฺ ผู้ทรงกรุณาปราณี ผู้ทรงเม...
1,"ALL PRAISE BE to Allah, Lord of all the worlds,",การสรรเสริญทั้งหลายนั้น เป็นสิทธิของอัลลอฮฺผู้...
2,"Most beneficent, ever-merciful,",ผู้ทรงกรุณาปราณี ผู้ทรงเมตตาเสมอ
3,King of the Day of Judgement.,ผู้ทรงอภิสิทธิ์แห่งวันตอบแทน
4,"You alone we worship, and to You alone turn fo...",เฉพาะพระองค์เท่านั้นที่พวกข้าพระองค์เคารพอิบาด...
...,...,...
93536,"The god (or judge) of Mankind,-",พระเป็นเจ้าแห่งมนุษย์ชาติ
93537,"From the mischief of the Whisperer (of Evil), ...",ให้พ้นจากความชั่วร้ายของผู้กระซิบกระซาบที่หลอกล่อ
93538,(The same) who whispers into the hearts of Man...,ที่กระซิบกระซาบในหัวอกของมนุษย์
93539,Among Jinns and among men.,จากหมู่ญินและมนุษย์


## Tatoeba

In [ ]:
path_en = '/content/TrainSet/OPUS_BiTextCollections/Tatoeba/Tatoeba.en-th.en'
path_th = '/content/TrainSet/OPUS_BiTextCollections/Tatoeba/Tatoeba.en-th.th'
tatoeba_df = pd.DataFrame(extract_data(path_en), columns = ['en_text'])
tatoeba_df['th_text'] = extract_data(path_th)
tatoeba_df

,en_text,th_text
0,Let's try something.,ลองทำดู!
1,Let's try something.,มาลองบางอย่างกันเถอะ
2,Let's try something.,ลองทำอะไรสักอย่างกัน
3,I have to go to sleep.,ฉันต้องไปนอนแล้ว
4,Today is June 18th and it is Muiriel's birthday!,วันนี้วันที่ 18 มิถุนายน และมันก็เป็นวันเกิดขอ...
...,...,...
1077,Who does your laundry for you?,ใครซักเสื้อผ้าของคุณให้คุณ
1078,How much would you charge me to do my laundry ...,คุณคิดค่าบริการฉันเท่าไรสำหรับการซักรีดให้ฉัน
1079,Do you do your own laundry?,คุณซักรีดเสื้อผ้าของตัวเองหรือไม่?
1080,I'm looking for someone to do my laundry for me.,ฉันกำลังมองหาคนซักเสื้อผ้าสำหรับฉัน


## Ubuntu Datasets

In [ ]:
path_en = '/content/TrainSet/OPUS_BiTextCollections/Ubuntu/Ubuntu.en-th.en'
path_th = '/content/TrainSet/OPUS_BiTextCollections/Ubuntu/Ubuntu.en-th.th'
Ubuntu_df = pd.DataFrame(extract_data(path_en), columns = ['en_text'])
Ubuntu_df['th_text'] = extract_data(path_th)
Ubuntu_df

,en_text,th_text
0,locale noexpr,locale noexpr
1,locale yesexpr,locale yesexpr
2,"--help"" and ""--version","--help"" and ""--version"
3,AisleRiot Solitaire,ถอดไพ่ AisleRiot
4,Play many different solitaire games,เล่นเกมถอดไพ่สารพัดแบบ
...,...,...
3781,Set a horizontal layout,ใช้การจัดวางแนวนอน
3782,_Vertical Layout,จัดวางแนว_ตั้ง
3783,Set a vertical layout,ใช้การจัดวางแนวตั้ง
3784,Click to close the side pane,คลิกเพื่อปิดช่องด้านข้าง


## SCB MT DATASETS

In [ ]:
apdf_df =pd.read_csv('/content/scb-mt-en-th-2020/apdf.csv')
apdf_df

,en_text,th_text
0,FAR LEFT: Indonesian National Police Chief Tit...,(ซ้ายสุด) นายติโต คาร์นาเวียน ผู้บัญชาการตำรวจ...
1,(Pictured: A rocket believed to be a Hwasong m...,(ภาพ: จรวดซึ่งเชื่อว่าเป็นขีปนาวุธฮวาซองเช่นเด...
2,"(Pictured: From left, Indian Air Commander Ak ...",(ภาพ: จากซ้าย นายอัค โชราเชีย ผู้บัญชาการทางอา...
3,"(Pictured: Left to right, Australia’s Defence ...",(ภาพ: จากซ้ายไปขวา นางมารีส เพย์น รัฐมนตรีว่าก...
4,(Pictured: A man watches the live-stream trans...,(ภาพ: ชายคนหนึ่งชมการแพร่ภาพออกอากาศสดการปล่อย...
...,...,...
13498,“It is not easy because we are 10 different co...,“ไม่ใช่เรื่องง่ายเลย เพราะเราเป็น 10 ประเทศที่...
13499,“It is not always that we’ve had the privilege...,“ไม่ได้มีบ่อยครั้งที่เรามีโอกาสพิเศษที่จะได้ให...
13500,"“There haven’t been many problems so far,” Kim...",“ไม่ได้มีปัญหามากนักจนถึงขณะนี้” นายคิมกล่าวสร...
13501,"“To fill the gap, more NCOs and COs can be a g...",” การเพิ่มจำนวนนายทหารชั้นประทวนและนายทหารสัญญ...


In [ ]:
assgov_df =pd.read_csv('/content/scb-mt-en-th-2020/assorted_government.csv')
assgov_df

,en_text,th_text
0,Reinvigorating Technical Cooperation for Susta...,เสริมสร้างความร่วมมือทางวิชาการสำหรับการพัฒนาอ...
1,Strengthening Cooperation for Peace and Security,เสริมสร้างความร่วมมือเพื่อสันติภาพและความมั่นคง
2,Engaging the Region and the World,ปฏิสัมพันธ์กับภูมิภาคและโลก
3,Advancing Thailand – Viet Nam Strategic Partne...,ส่งเสริมความเป็นหุ้นส่วนทางยุทธศาสตร์ระหว่างไท...
4,4. Further inquiries can be directed to:,๔. หากมีข้อสอบถามเพิ่มเติม โปรดติดต่อได้ที่
...,...,...
25393,If offender in accordance with first paragraph...,ถ้าผู้กระทำความผิดตามวรรคแรกหรือวรรคสองเป็นผู้...
25394,Offences Relating to Cause Public Dangers as p...,(1) ความผิดเกี่ยวกับการก่อให้เกิดภยันตรายต่อปร...
25395,If the offence being committed under any of th...,ถ้าความผิดนั้นมีลักษณะประการหนึ่งประการใดดังที...
25396,In determination of period of the hour time on...,ในการกำหนดระยะเวลาทำงานแทนค่าปรับตามวรรคสาม ให...


In [ ]:
wiki_df = pd.read_csv('/content/scb-mt-en-th-2020/wikipedia.csv')
wiki_df

,en_text,th_text
0,Palestine at the 2004 Summer Olympics,ปาเลสไตน์ในโอลิมปิกฤดูร้อน 2004
1,Palestine competed at the 2004 Summer Olympics...,ปาเลสไตน์ในโอลิมปิกฤดูร้อน 2004 ปาเลสไตน์ เข้า...
2,Pakistan at the 2004 Summer Olympics,ประเทศปากีสถานในโอลิมปิกฤดูร้อน 2004
3,Pakistan competed at the 2004 Summer Olympics ...,ประเทศปากีสถานในโอลิมปิกฤดูร้อน 2004 ประเทศปาก...
4,Sri Lanka at the 2004 Summer Olympics,ประเทศศรีลังกาในโอลิมปิกฤดูร้อน 2004
...,...,...
33751,1968 in Portugal Events in the year 1968 in Po...,ประเทศโปรตุเกสใน ค.ศ. 1968 เหตุการณ์ที่เกิดขึ้...
33752,The following lists events that happened durin...,ประเทศโปรตุเกสใน ค.ศ. 2000 เหตุการณ์ที่เกิดขึ้...
33753,1994 in Portugal Events in the year 1994 in Po...,ประเทศโปรตุเกสใน ค.ศ. 1994 เหตุการณ์ที่เกิดขึ้...
33754,1998 in Portugal Events in the year 1998 in Po...,ประเทศโปรตุเกสใน ค.ศ. 1998 เหตุการณ์ที่เกิดขึ้...


In [ ]:
thweb_df = pd.read_csv('/content/scb-mt-en-th-2020/thai_websites.csv')
thweb_df

,en_text,th_text
0,CLARINS This intensive replenishing cream hel...,คืนความยืดหยุ่นให้ผิวที่ร่วงโรยตามวัย จากการเป...
1,ONLY@CENTRAL Color : Almond Size : 32 A UK Ca...,ONLY@CENTRAL สี : เบจไซส์ : 32 A UK หมายเหตุ:...
2,ONLY@CENTRAL Color : Almond Size : 32 B UK Ca...,ONLY@CENTRAL สี : เบจไซส์ : 32 B UK หมายเหตุ:...
3,ONLY@CENTRAL Color : Almond Size : 32 C UK Ca...,ONLY@CENTRAL สี : เบจไซส์ : 32 C UK หมายเหตุ:...
4,ONLY@CENTRAL Color : Almond Size : 34 A UK Ca...,ONLY@CENTRAL สี : เบจไซส์ : 34 A UK หมายเหตุ:...
...,...,...
120275,・Clear Lotion：After face washing apply an appr...,・เคลียร์โลชั่น：ใช้หลังล้างหน้า เทลงบนสำลีแผ่นพ...
120276,﻿﻿﻿Here is a charming take-along castle perfec...,ชุดปราสาทเจ้าหญิง จาก MELISSA & DOUG ที่มาพร้อ...
120277,﻿﻿﻿﻿﻿Give little builders all the tools they n...,ช่างก่อสร้างตัวน้อยกับไอเดียสุดเจ๋ง จะทำให้งาน...
120278,﻿﻿﻿﻿﻿﻿Everyone likes getting letters! Sending ...,เล่นสวมบทบาทเป็นบุรุษไปรษณีย์ ด้วยชุดของเล่นส่...


In [ ]:
tskmaster_df = pd.read_csv('/content/scb-mt-en-th-2020/task_master_1.csv')
tskmaster_df

,en_text,th_text
0,"Hi, I'm looking to book a table for Korean fod.",สวัสดีค่ะ ช่วยจองร้านอาหารเกาหลีให้หน่อยได้มั้...
1,"Ok, what area are you thinking about?",ได้เลยค่ะ แถวไหนดีคะ?
2,"Somewhere in Southern NYC, maybe the East Vill...",แถว ๆ นิวยอร์คทางใต้ก็ได้ค่ะ แถวอีสต์วิลเลจอะไ...
3,"Ok, great. There's Thursday Kitchen, it has gr...",โอเคค่ะ ดีเลย มีร้านเธอร์เดย์คิทเช่นอยู่ รีวิว...
4,That's great. So I need a table for tonight at...,เยี่ยมค่ะ อยากได้โต๊ะแปดคนตอนทุ่มนึงค่ะ นั่งตร...
...,...,...
222728,did you want thick crust for all 3 pizzas?,ขอบหนาทั้ง 3 ถาดเลยมั้ยคะ
222729,"you said you want to order 3 large pizzas, all...",สรุปเป็นพิซซ่าไซส์ใหญ่สามถาดนะคะ ทุกถาดพิเศษชี...
222730,the total for your order is $56.44.,ราคาทั้งหมด 56.44 เหรียญค่ะ
222731,"Yes, can you make that for pickup, please?",ค่ะ เป็นออเดอร์สำหรับไปรับที่ร้านนะคะ


In [ ]:
paracrawl_df = pd.read_csv('/content/scb-mt-en-th-2020/paracrawl.csv')
paracrawl_df

,en_text,th_text
0,We are open to new proposals.,เราจะเปิดให้ข้อเสนอใหม่
1,Big lottery jackpots in the rear,รางวัลจับสลากใหญ่ในด้านหลัง
2,Car rental tips for Bangkok Airport,เคล็ดลับการเช่ารถสำหรับ กรุงเทพมหานคร ท่าอากาศยาน
3,visible structure moving along the wall.,โครงสร้างที่มองเห็นได้ย้ายตามผนัง
4,Price List by Duration of Stay,รายการค่าใช้จ่ายตามระยะเวลาของคอร์ส
...,...,...
60034,Where are the last two weeks in advance? Our t...,ที่ไหนในสองสัปดาห์ที่ผ่านมาจะออก? เวลาของเราใน...
60035,"After a good night we set her on the van, invi...","หลังจากที่พักผ่อนสบายตลอดคืนที่เราไปมาทั่วแวน,..."
60036,"The question is, what is better - or granite t...",คำถามคือสิ่งที่ดีกว่า - หินแกรนิตหรือกระเบื้อง...
60037,Matches Fashion Luxury Shopping in the event y...,ตรงกับแฟชั่นหรูหราชอปปิ้งอยู่ในเหตุการณ์คุณยัง...


In [ ]:
nussms_df = pd.read_csv('/content/scb-mt-en-th-2020/nus_sms.csv')
nussms_df

,en_text,th_text
0,"He is out for work, donot disturb. kindly call...",เขาไปทำธุระข้างนอก กรุณาโทรหาเขา 5 นาทีหลังจาก...
1,Huh u still doing yest tut... I tot u finish l...,นี่ นายทำไปถีงไหนแล้ว ฉันคิดว่านายทำเสร็จแล้วซ...
2,No need 2 b paisei w me lar... U mus eat more ...,ไม่ต้องมาทำเป็นอายอะไรกับฉันนักหรอก แกกินเข้าไ...
3,Yar lor... U not ard then i v lonely lor... He...,กินเถอะ เธอไม่กิน ฉันก็เหงาแย่สิ นี่ เธอไม่กิน...
4,Of course got use lah. U must jia you k. 2 mor...,ตกลงตามนั้นน่ะ นายจะสนุกมาก ถ้าเพิ่มอีกสัก 2 วัน
...,...,...
43745,mornin. u n jessen on rite? i mean on chris ev...,หวัดดีตอนเช้า เธอกับเจนเซ่นยังมาใช่ไหม ฉันหมาย...
43746,"you always talk-cock, so black-listed",เธอชอบพูดไร้สาระ เลยโดนขึ้นบัญชีดำ
43747,re 8th out of 16 teams,ลำดับที่ 8 จากทั้งหมด 16 ทีม
43748,"s great man,you keep up the good work yeah",นายยอดมาก ยังรักษาผลงานได้ดีเสมอ


In [ ]:
msr_df = pd.read_csv('/content/scb-mt-en-th-2020/msr_paraphrase.csv')
msr_df

,en_text,th_text
0,"Just as before, it will be up to the Council t...",เช่นเดียวกับเมื่อก่อน สภาเป็นผู้ที่จะตัดสินใจท...
1,"In December, he had predicted a 5 percent grow...",ในเดือนธันวาคม เขาคาดว่าจะมีอัตราการเติบโต 5 เ...
2,"Character actor Hume Cronyn, 91, died Sunday a...",นักแสดงระดับตำนาน ฮูม โครนีน เสียชีวิตในวัย 91...
3,"On May 22, 2002, a man walking his dog came ac...",เมื่อวันที่ 22 พฤษภาคม 2002 ชายคนหนึ่งพบกระดูก...
4,The bodies of 18 illegal Mexican immigrants wh...,ร่างของผู้อพยพชาวเม็กซิกัน 18 คนที่เสียชีวิตเน...
...,...,...
10366,"""Events on our system, in and of themselves, c...",“สิ่งที่เกิดขึ้นในระบบของเรา ทั้งในและด้วยตัวข...
10367,"Handset market share for the second quarter, i...",ส่วนแบ่งการตลาดของอุปกรณ์พกพาในไตรมาสที่สอง มั...
10368,An attempt last month in the Senate to keep th...,ความพยายามในเดือนที่ผ่านมาในวุฒิสภาเพื่อให้กอง...
10369,"Writing for the minority, Justice John Paul St...",เขียนเพื่อชนกลุ่มน้อย ผู้พิพากษา John Paul Ste...


In [ ]:
mcv_df = pd.read_csv('/content/scb-mt-en-th-2020/mozilla_common_voice.csv')
mcv_df

,en_text,th_text
0,"The fool wanders, the wise man travels.",คนโง่พเนจร คนฉลาดท่องเที่ยว
1,One of these days is none of these days.,หนึ่งในวันเหล่านี้คือไม่มีวันเหล่านี้เลย
2,"Necessity is a hard nurse, but she raises stro...","ความจำเป็นเป็นสิ่งที่ยาก, แต่เธอสามารถเลี้ยงลู..."
3,In one ear and out the other.,เข้าหู้ข้างหนึ่งและออกอีกข้างหนึ่ง
4,It can't happen here is number one on the list...,ไม่สามารถเกิดขึ้นที่นี่ได้คือคำพูดสุดท้ายที่ดั...
...,...,...
33792,"If you can't help, don't hinder.",หากคุณไม่สามารถช่วยได้อย่าขัดขวาง
33793,It's all in a days work.,ทำงานทั้งวัน
33794,Laziness travels so slowly that poverty soon o...,ความเกียจคร้านเดินทางช้ามากจนความยากจนมาถึงในไ...
33795,Pushchairs can be folded when the toddler want...,สามารถพับเก็บเก้าอี้ลงได้เมื่อทารกต้องการเดิน


In [ ]:
gen_yn = pd.read_csv('/content/scb-mt-en-th-2020/generated_reviews_yn.csv')
gen_yn

,en_text,th_text
0,We are trying to use them on our Samsung Smart...,เรากำลังพยายามใช้พวกเขาบน Samsung Smart TV ของเรา
1,This thing will not work with Mac OSX 10.4.7.1.,สิ่งนี้จะไม่ทำงานกับ Mac OSX 10.4.7.1
2,We are very happy with our Dyson DC25.,เรามีความสุขมากกับ Dyson DC25 ของเรา
3,It doesn't work with Skype.,มันไม่ทำงานกับ Skype
4,I'll be looking at Cisco next.,ฉันจะดู Cisco ต่อไป
...,...,...
280203,Disappointed also when my credit card company ...,ผิดหวังเมื่อ บริษัท บัตรเครดิตของฉันปฏิเสธการค...
280204,"No explanation, just no response.",ไม่มีคําอธิบายไม่มีคําตอบ
280205,Not happy at all with this experience.,ไม่มีความสุขเลยกับประสบการณ์นี้
280206,Will stay away from ordering books on Amazon i...,จะอยู่ห่างจากการสั่งซื้อหนังสือใน Amazon ในอนาคต


In [ ]:
gen_trans = pd.read_csv('/content/scb-mt-en-th-2020/generated_reviews_translator.csv')
gen_trans

,en_text,th_text
0,These are a terrible product from China.The li...,สินค้าพวกนี้เป็นสินค้าห่วยๆที่มาจากประเทศจีน ท...
1,They can't be used for anything other than par...,ไม่สามารถใช้ทำอะไรได้มากกว่าใช้ประดับในงานเลี้ยง
2,"And yes, I followed directions exactly as they...",และแน่นอนว่าฉันได้ทำตามแนวทางที่เขาเขียนไว้อย่...
3,What kind of trash can cannot survive simple h...,กระป๋องขยะประเภทไหนกันที่ไม่สามารถใช้งานง่ายๆใ...
4,I'm returning them.,ฉันจะส่งสินค้าพวกนี้กลับค่ะ
...,...,...
133325,The songs are well known and the sound quality...,เพลงเป็นที่รู้จักกันดีและคุณภาพเสียงก็ยอดเยี่ยม
133326,I think it might be more important these days ...,ฉันคิดว่ามันอาจจะสำคัญกว่าในทุกวันนี้ที่จะกลับ...
133327,"If that's not what you want, buy this CD.",ถ้านั่นไม่ใช่สิ่งที่คุณต้องการซื้อซีดีนี้
133328,It brings me back 20 years when we used to lis...,มันทำให้ฉันย้อนกลับไป 20 ปีเมื่อเราเคยฟัง John...


In [ ]:
gen_crowd = pd.read_csv('/content/scb-mt-en-th-2020/generated_reviews_crowd.csv')
gen_crowd

,en_text,th_text
0,Doesn't snap together well.,เข้ากันไม่ค่อยดี
1,Charged it after every use as directed for abo...,เรียกเก็บเงินหลังจากใช้งานทุกครั้งตามที่กำกับไ...
2,"My son wanted this movie so badly, that he sai...",ลูกชายของฉันต้องการภาพยนตร์เรื่องนี้เพื่อไม่ดี...
3,But his writing has degenerated in later books.,แต่หนังสือเล่มที่ผ่านมาของเขามันดูด้อยคุณภาพลง
4,"It was supposed to be a good bag, well you get...",มันควรที่จะเป็นกระเป๋าที่ดี เอ้อ เธอได้ในสิ่งท...
...,...,...
24582,This stuff just tastes awful.,สินค้าชิ้นนี้แย่มาก
24583,I would not have bought this if my son hadn't ...,ฉันไม่ชอบซื้อเวลาที่ลูกชายทำไม่เสร็จ
24584,But when you end up with a tape full of shows ...,แต่เมื่อคุณจบลงด้วยเทปเต็มรูปแบบของการแสดงเช่น...
24585,"But the ""e-version"" is very hard to use.",แต่เวอร์ชั่น E ใช้งานยาก


Running the code above will write the two files, train.tsv and eval.tsv, to the data/ directory.

# Convert to MT5 STF Format Function

## All Data

In [ ]:
frames = [lower_case_df] #Ubuntu_df , qed_df, tanzil_df, tatoeba_df,  Bible_df, GNOME_df, JW300_df, opensub_df
all_df = pd.concat(frames)

In [ ]:
def convert_df_to_format(input_df):
  data = []
  for row, series in input_df.iterrows():
    data.append(["translate thai to english", series['th_text'], series['en_text']])
    # data.append(["translate english to thai", series['en_text'], series['th_text']])
  output_df = pd.DataFrame(data, columns=["prefix", "input_text", "target_text"])
  return output_df

In [ ]:
# Test convert function
all_formatted = convert_df_to_format(all_df)
all_formatted

,prefix,input_text,target_text
0,translate thai to english,ขอบคุณ แต่ ฉัน ไม่ มี เวลา พอ,thank you but i don't have enough time
1,translate thai to english,"โอ้ , คุณ รู้ , เมื่อ ฉัน ตื่น ฉัน จะ ออกไป เด...","oh, you know, when i get up i usually do my st..."
2,translate thai to english,หน้าที่ ของ ตำรวจ คือ การ ปกป้อง ประชาชน,the duty of a policeman is to safeguard people .
3,translate thai to english,มัน เป็นความ โอหัง ของ เพศ หญิง .,it's female vanity.
4,translate thai to english,โปรด คอย ดู อย่า เผลอ,please keep a sharp lookout .
...,...,...,...
681534,translate thai to english,ตอน วิจัย พูด โทรศัพท์ กับ ดิฉัน เสียง เขา แตก...,"when wijai talked to me on the phone , his voi..."
681535,translate thai to english,สถานี รถ โดยสาร ทาง ตะวันตก ของ เมือง,the bus terminal is on the west side of the ci...
681536,translate thai to english,ลักษณะ ภาย นอก,encapsulate
681537,translate thai to english,ความ ตาย ทำ ให้ ผม กลัว โดย เฉพาะอย่างยิ่ง การ...,"death frightens me , specifically my own death ."


In [ ]:
# all_formatted1 = all_formatted
# all_formatted1

## Model Load

In [ ]:
import logging
import pandas as pd
from simpletransformers.t5 import T5Model, T5Args


logging.basicConfig(level=logging.INFO)
transformers_logger = logging.getLogger("transformers")
transformers_logger.setLevel(logging.WARNING)

# Model Training

Once we have the data files, we are ready to start training the model.

First, we’ll import the necessary stuff and set up logging.


Next, we set up our training and evaluation data.

Here, we remove the prefix values from the datasets because we expect the model to infer the required task based on the input. If the input is in English, then it should be translated to Sinhalese. If it’s in Sinhalese, then it should be translated to English. The model shouldn’t need a prefix to figure this out after the training!

You can use a prefix value to tell an mT5 (or T5) to perform a specific task. This is quite useful to train a model which can perform multiple tasks, as shown in the article below.

**Sidenote on GPU memory usage**

The amount of GPU memory required to train a Transformer model depends on many different factors (maximum sequence length, number of layers, number of attention heads, size of the hidden dimensions, size of the vocabulary, etc.). Out of these, the maximum sequence length of the model is one of the most significant.

For the self-attention mechanisms used in the mT5 model, the memory requirement grows quadratically with the input sequence length (O(n²) space complexity). I.e., When the sequence length doubles, the memory required quadruples.

Also, mT5 has a much larger vocabulary than T5 (~250,000 tokens to ~32,000 tokens), contributing to mT5 being quite punishing in terms of GPU memory required.

The takeaway from all this is that the number of tokens we can input to the model (the maximum sequence length) comes at a hefty premium. Based on this, it’s wasteful to use even a small number of tokens on the prefix if the model can do without.

Now, let’s get back to training the model!

In [ ]:
from google.colab import drive
drive.mount('/content/mydrive/')

Drive already mounted at /content/mydrive/; to attempt to forcibly remount, call drive.mount("/content/mydrive/", force_remount=True).


In [ ]:
model_args = T5Args()
model_args.max_seq_length = 96
model_args.train_batch_size = 8
#model_args.eval_batch_size = 4
model_args.num_train_epochs = 1
model_args.evaluate_during_training = False
#model_args.evaluate_during_training_steps = 200
model_args.use_multiprocessing = False
model_args.fp16 = False
model_args.save_steps = -1
model_args.save_eval_checkpoints = False
model_args.no_cache = True
model_args.reprocess_input_data = True
model_args.overwrite_output_dir = True
model_args.preprocess_inputs = False
model_args.num_return_sequences = 1
#model_args.wandb_project = "MT5 Sinhala-English Translation"

model = T5Model("mt5", "google/mt5-base", args=model_args)

Downloading:   0%|          | 0.00/702 [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/2.17G [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/4.11M [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/376 [00:00<?, ?B/s]

Here, we specify how we want the model to be set up and initialize the pre-trained mT5 model according to model_args.

I’m using a maximum sequence length (max_seq_length) of 96 and a train/eval batch sizes of 20. Generally, larger batch sizes mean better GPU utilization, and therefore, shorter training times. As mentioned earlier, longer sequences require more GPU memory, which means smaller batch sizes and longer training times. The maximum sequence length of 96 allows the model to work with reasonably long text (typically a few sentences) while also keeping the training time practical.

Note that you may need to tweak these values to train the model on your own GPU. If you run out of GPU memory (CUDA memory error), try reducing the batch sizes and/or the maximum sequence length.

If you want to try the fine-tuned model, you can find it here on the Huggingface model hub.

Now, to run the training, we just need to call the train_model() method.

In [ ]:
type(train_df)

pandas.core.frame.DataFrame

In [ ]:
# Train the modelllllll
%%timeit
model.train_model(all_formatted)  #(train_df, eval_data=eval_df)

INFO:simpletransformers.t5.t5_utils: Creating features from dataset file at cache_dir/


  0%|          | 0/681539 [00:00<?, ?it/s]

/usr/local/lib/python3.7/dist-packages/transformers/tokenization_utils_base.py:3524: FutureWarning: 
`prepare_seq2seq_batch` is deprecated and will be removed in version 5 of HuggingFace Transformers. Use the regular
`__call__` method to prepare your inputs and the tokenizer under the `as_target_tokenizer` context manager to prepare
your targets.

Here is a short example:

model_inputs = tokenizer(src_texts, ...)
with tokenizer.as_target_tokenizer():
    labels = tokenizer(tgt_texts, ...)
model_inputs["labels"] = labels["input_ids"]

See the documentation of your specific tokenizer for more details on the specific arguments to the tokenizer of choice.
For a more complete example, see the implementation of `prepare_seq2seq_batch`.

  warnings.warn(formatted_warning, FutureWarning)
INFO:simpletransformers.t5.t5_model: Training started


Epoch:   0%|          | 0/1 [00:00<?, ?it/s]

Running Epoch 0 of 1:   0%|          | 0/85193 [00:00<?, ?it/s]

...

  0%|          | 0/681539 [00:00<?, ?it/s]

INFO:simpletransformers.t5.t5_model: Training started


Epoch:   0%|          | 0/1 [00:00<?, ?it/s]

Running Epoch 0 of 1:   0%|          | 0/85193 [00:00<?, ?it/s]

KeyboardInterrupt: ignored

In [ ]:
!zip -r /content/best_model_th_en.zip /content/outputs/

In [ ]:
!cp /content/best_model_th_en.zip /content/mydrive/MyDrive/MT-Hackathon/best_model_th_en.zip

## Train another epochs

In [ ]:
model_args = T5Args()
model_args.max_seq_length = 96
model_args.train_batch_size = 8
#model_args.eval_batch_size = 4
model_args.num_train_epochs = 2
model_args.evaluate_during_training = False
#model_args.evaluate_during_training_steps = 200
model_args.use_multiprocessing = False
model_args.fp16 = False
model_args.save_steps = -1
model_args.save_eval_checkpoints = False
model_args.no_cache = True
model_args.reprocess_input_data = True
model_args.overwrite_output_dir = True
model_args.preprocess_inputs = False
model_args.num_return_sequences = 1
#model_args.wandb_project = "MT5 Sinhala-English Translation"

model = T5Model("mt5", "/content/content/outputs/", args=model_args)

In [ ]:
%%timeit
model.train_model(all_formatted)  #(train_df, eval_data=eval_df)

INFO:simpletransformers.t5.t5_utils: Creating features from dataset file at cache_dir/


  0%|          | 0/681539 [00:00<?, ?it/s]

/usr/local/lib/python3.7/dist-packages/transformers/tokenization_utils_base.py:3524: FutureWarning: 
`prepare_seq2seq_batch` is deprecated and will be removed in version 5 of HuggingFace Transformers. Use the regular
`__call__` method to prepare your inputs and the tokenizer under the `as_target_tokenizer` context manager to prepare
your targets.

Here is a short example:

model_inputs = tokenizer(src_texts, ...)
with tokenizer.as_target_tokenizer():
    labels = tokenizer(tgt_texts, ...)
model_inputs["labels"] = labels["input_ids"]

See the documentation of your specific tokenizer for more details on the specific arguments to the tokenizer of choice.
For a more complete example, see the implementation of `prepare_seq2seq_batch`.

  warnings.warn(formatted_warning, FutureWarning)
INFO:simpletransformers.t5.t5_model: Training started


Epoch:   0%|          | 0/2 [00:00<?, ?it/s]

INFO:simpletransformers.t5.t5_model:   Continuing training from checkpoint, will skip to saved global_step
INFO:simpletransformers.t5.t5_model:   Continuing training from epoch 1
INFO:simpletransformers.t5.t5_model:   Continuing training from global step 85193
INFO:simpletransformers.t5.t5_model:   Will skip the first 0 steps in the current epoch


Running Epoch 0 of 2:   0%|          | 0/85193 [00:00<?, ?it/s]

In [ ]:
!zip -r /content/best_model_th_en2.zip /content/outputs/

In [ ]:
!cp /content/best_model_th_en2.zip /content/mydrive/MyDrive/MT-Hackathon/best_model_th_en2.zip

# Loading Model

In [ ]:
!pip install --upgrade --no-cache-dir gdown

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
    Preparing wheel metadata ... done
  Created wheel for gdown: filename=gdown-4.4.0-py3-none-any.whl size=14774 sha256=31017d14ff8f55fddbff081e2388ad21bd82c69d61c39b9aa867cfc647cf6aa2
  Stored in directory: /tmp/pip-ephem-wheel-cache-_4pepnkj/wheels/fb/c3/0e/c4d8ff8bfcb0461afff199471449f642179b74968c15b7a69c
Successfully built gdown
  Attempting uninstall: gdown
    Found existing installation: gdown 4.2.2
    Uninstalling gdown-4.2.2:
      Successfully uninstalled gdown-4.2.2


In [ ]:
!gdown --id 1gbPxvhhRmfYWl3AAu1gWtibetFAUF-7P

/usr/local/lib/python3.7/dist-packages/gdown/cli.py:131: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  category=FutureWarning,
Downloading...
From: https://drive.google.com/uc?id=1gbPxvhhRmfYWl3AAu1gWtibetFAUF-7P
To: /content/best_model_th_en.zip
100% 3.74G/3.74G [00:17<00:00, 214MB/s]


In [ ]:
!unzip -qq /content/best_model_th_en.zip

In [ ]:
model_args = T5Args()
model_args.max_seq_length = 96
model_args.train_batch_size = 8
#model_args.eval_batch_size = 4
model_args.num_train_epochs = 1
model_args.evaluate_during_training = False
#model_args.evaluate_during_training_steps = 200
model_args.use_multiprocessing = False
model_args.fp16 = False
model_args.save_steps = -1
model_args.save_eval_checkpoints = False
model_args.no_cache = True
model_args.reprocess_input_data = True
model_args.overwrite_output_dir = True
model_args.preprocess_inputs = False
model_args.num_return_sequences = 1
model_args.use_cuda = False
#model_args.wandb_project = "MT5 Sinhala-English Translation"

model = T5Model("mt5", "/content/content/outputs/checkpoint-85193-epoch-1/", args=model_args)

ValueError: ignored

# Validation

## Evaluating the Model

The standard metric used to evaluate and compare machine translation models is the BLEU score, specifically, the BLEU scheme used by the annual Conference on Machine Translation (WMT). The SacreBLEU library can be used to calculate this score.

For more information on the BLEU score, please refer to this paper by Matt Post.

Since The Tatoeba Challenge also provides the BLEU scores for the benchmark translation models, we can easily compare our model to the benchmark model.

Now, let’s load our fine-tuned model and see how it stacks up!

In [ ]:
!pip install sacrebleu

     |████████████████████████████████| 90 kB 8.8 MB/s 


In [ ]:
import logging
# import sacrebleu
import pandas as pd
from simpletransformers.t5 import T5Model, T5Args


logging.basicConfig(level=logging.INFO)
transformers_logger = logging.getLogger("transformers")
transformers_logger.setLevel(logging.WARNING)


model_args = T5Args()
model_args.max_length = 96
model_args.length_penalty = 1
model_args.num_beams = 10
model_args.use_cuda = False

model = T5Model("mt5", "/content/content/outputs/", args=model_args)

ValueError: ignored

We import the necessary stuff (note the sacrebleu library) and initialize the model just as we did for training, except that we load the fine-tuned model from outputs/ rather than the pre-trained model.

We also set some parameters for generating text (decoding) with the model. Here, the max_length is the maximum length for the output from the model rather than the input.

If you’d like to learn more about the decoding process, please refer to the decoding algorithms section in this article and this excellent notebook by Huggingface.

Next, we’ll prepare the data for evaluation.

## Validation Datasets

In [ ]:
path_en = '/content/ValidationSet/Valid1+2/dev.en'
path_th = '/content/ValidationSet/Valid1+2/dev.th'
valid1_df = pd.DataFrame(extract_data(path_en), columns = ['en_text'])
valid1_df['th_text'] = extract_data(path_th)
valid1_df

,en_text,th_text
0,It has been confirmed that eight thoroughbred ...,ได้ เป็น ที่ ยืนยัน แน่นอน แล้ว ว่า ม้า แข่ง พ...
1,"Randwick has been locked down, and is expected...",สนาม แข่ง ม้า แร ็นด์ วิค ถูก ปิด และ คาด ว่า ...
2,It is expected that the virulent flu will affe...,คาด ว่า ไข้ หวัด ใหญ่ ร้ายแรง นี้ จะ กระทบ ต่อ...
3,NSW Minister for Primary Industries said the f...,รัฐมนตรี อุตสาหกรรม พื้นฐาน นอง รัฐNSW กล่าว ว...
4,The cases are the first infections of race hor...,เหตุการณ์ นี้ คือ การ ติด เชื้อ ครั้ง แรก ใน ม...
...,...,...
1014,They are reported to have verified her identit...,มี รายงาน แจ้ง ว่า พวก เขา ทราบ ตัว ตน ของ เธอ...
1015,TMZ.com reports that police were checking to s...,TMZ.com รายงาน ว่า ทาง ตำรวจ ได้ ตรวจสอบ ใบ อน...
1016,Spears was not charged with anything and despi...,Spears ไม่ ถูก ปรับ ใน ข้อ หา ใด ๆ และ แม้ ว่า...
1017,"""Britney Spears was part of the group, but was...",SaraFaden โฆษก หญิง ของ กรมตำรวจ LosAngeles กล...


In [ ]:
path_en = '/content/ValidationSet/Valid1+2/dev2.en'
path_th = '/content/ValidationSet/Valid1+2/dev2.th'
valid2_df = pd.DataFrame(extract_data(path_en), columns = ['en_text'])
valid2_df['th_text'] = extract_data(path_th)
valid2_df

,en_text,th_text
0,Did you get that thesis from the Dude?,คุณได้ข้อเสนอนี้ จากเพื่อนคนนั้นเหรอ?
1,Police! Get your hands up!,มันเป็นช่วงนาทีที่ ดึงคุณออกจากตัวคุณเอง
2,"We should eat. You hungry, buddy? Yeah.",กินกันเถอะ หิวแล้วใช่มั้ย
3,I'd watch myself if I were you. (Coughs) Ok.,ถ้าฉันเป็นนาย จะคอยระวังตัวไว้ โอเค ที่นี่ล่ะ
4,Anyhow ... it appears you left a favorable imp...,ยังไงก็ตามดูเหมือนว่าคุณทำให้ที่ประชุมวันนี้ ป...
...,...,...
1996,We must close the gate.,เราจะต้องปิดประตู.
1997,You said I was free to come and go.,คุณพูดเองว่าฉันมีสิทธิ์ที่จะไปไหนมาไหนก็ได้
1998,"Well, we're on pins and needles.",เหน็บกินแล้วนะ
1999,"Kjetil, we have to get up. We have to get up.",เชทิล ตื่นเถอะ


In [ ]:
frames = [valid1_df]  # valid2_df # valid1_df
valid_df = pd.concat(frames, ignore_index = True)
valid_df = valid_df[:1018]
valid_df

,en_text,th_text
0,It has been confirmed that eight thoroughbred ...,ได้ เป็น ที่ ยืนยัน แน่นอน แล้ว ว่า ม้า แข่ง พ...
1,"Randwick has been locked down, and is expected...",สนาม แข่ง ม้า แร ็นด์ วิค ถูก ปิด และ คาด ว่า ...
2,It is expected that the virulent flu will affe...,คาด ว่า ไข้ หวัด ใหญ่ ร้ายแรง นี้ จะ กระทบ ต่อ...
3,NSW Minister for Primary Industries said the f...,รัฐมนตรี อุตสาหกรรม พื้นฐาน นอง รัฐNSW กล่าว ว...
4,The cases are the first infections of race hor...,เหตุการณ์ นี้ คือ การ ติด เชื้อ ครั้ง แรก ใน ม...
...,...,...
1013,Police also say that at least one of cars foll...,ตำรวจ ยัง เสริม ว่า มี รถ อย่าง น้อย หนึ่ง คัน...
1014,They are reported to have verified her identit...,มี รายงาน แจ้ง ว่า พวก เขา ทราบ ตัว ตน ของ เธอ...
1015,TMZ.com reports that police were checking to s...,TMZ.com รายงาน ว่า ทาง ตำรวจ ได้ ตรวจสอบ ใบ อน...
1016,Spears was not charged with anything and despi...,Spears ไม่ ถูก ปรับ ใน ข้อ หา ใด ๆ และ แม้ ว่า...


In [ ]:
# eval_df = pd.read_csv("data/eval.tsv", sep="\t").astype(str)

# thai_truth = [valid_df.loc[valid_df["prefix"] == "translate english to thai"]["target_text"].tolist()]
# to_thai = valid_df.loc[valid_df["prefix"] == "translate english to english"]["input_text"].tolist()

english_truth = valid_df['en_text'].tolist()
to_english = valid_df['th_text'].tolist()

Here, we load the evaluation data and prepare separate lists of inputs and true translations (for English to Sinhalese and vice versa).

With the model and evaluation data loaded and ready, we can go ahead and do the translations. With Simple Transformers, we just call model.predict() with the input data. Then, we use the sacrebleu tool to calculate the BLEU score.

The sacrebleu library should be installed in your virtual environment if you followed the setup instructions. If not, you can install it with pip install sacrebleu.

In [ ]:
english_truth

In [ ]:
to_english

In [ ]:
def post_process(pred):
  post_eng_preds = []
  for sentence in pred:
    post_eng_pred = sentence.replace("&apos;", "'")
    post_eng_preds.append(post_eng_pred)

  return post_eng_preds

In [ ]:
len(english_preds)

3019

In [ ]:
len(english_truth)

3019

In [ ]:
english_preds = model.predict(to_english)

Generating outputs:   0%|          | 0/128 [00:00<?, ?it/s]

/usr/local/lib/python3.7/dist-packages/transformers/tokenization_utils_base.py:3524: FutureWarning: 
`prepare_seq2seq_batch` is deprecated and will be removed in version 5 of HuggingFace Transformers. Use the regular
`__call__` method to prepare your inputs and the tokenizer under the `as_target_tokenizer` context manager to prepare
your targets.

Here is a short example:

model_inputs = tokenizer(src_texts, ...)
with tokenizer.as_target_tokenizer():
    labels = tokenizer(tgt_texts, ...)
model_inputs["labels"] = labels["input_ids"]

See the documentation of your specific tokenizer for more details on the specific arguments to the tokenizer of choice.
For a more complete example, see the implementation of `prepare_seq2seq_batch`.

  warnings.warn(formatted_warning, FutureWarning)


Decoding outputs:   0%|          | 0/1018 [00:00<?, ?it/s]

In [ ]:
english_preds

['eight good racing horses are certainly immune to a big cold in manchester.',
 "radwick's horse racing arena was closed and expected to remain close to two months.",
 'it is supposed that this bad cold could affect a large number of seven large horses in the stables.',
 'ns ministers, nsa state, said, will be confined for five days after the last symptoms of a cold.',
 'this incident is the first infection in racing horses, including infections in horses, ns and queens and queens.',
 'this cold fever has an acute opportunity of infection, but could not breed humans.',
 "the country's horse racing costs several million dollars every day.",
 'the president of the horse\'s last weekend was blocked, while, was a day, " sorrow and gloomy horses.',
 "it's supposed to be there's a horse race around every australian states except ns and queens this weekend.",
 "sydney's new year's carnival racing festival was cancelled, but at melbourne was supposed to start this weekend.",
 'this tournament 

In [ ]:
th_eng_bleu = sacrebleu.corpus_bleu(english_preds, english_truth)
print("English to Thai: ", th_eng_bleu.score)

English to Thai:  0.10052311972059963


In [ ]:
english_preds_new = post_process(english_preds)

In [ ]:
english_preds_new

['eight good racing horses are certainly immune to a big cold in manchester.',
 "radwick's horse racing arena was closed and expected to remain close to two months.",
 'it is supposed that this bad cold could affect a large number of seven large horses in the stables.',
 'ns ministers, nsa state, said, will be confined for five days after the last symptoms of a cold.',
 'this incident is the first infection in racing horses, including infections in horses, ns and queens and queens.',
 'this cold fever has an acute opportunity of infection, but could not breed humans.',
 "the country's horse racing costs several million dollars every day.",
 'the president of the horse\'s last weekend was blocked, while, was a day, " sorrow and gloomy horses.',
 "it's supposed to be there's a horse race around every australian states except ns and queens this weekend.",
 "sydney's new year's carnival racing festival was cancelled, but at melbourne was supposed to start this weekend.",
 'this tournament 

In [ ]:
th_eng_bleu_new = sacrebleu.corpus_bleu(english_preds_new, english_truth)
print("Thai to English after Post-process: ", th_eng_bleu_new.score)

Thai to English after Post-process:  0.10052311972059963


In [ ]:
# #a_list = ["abc", "def", "ghi"]

# textfile = open("/content/th2en", "w")

# for element in english_preds:

#     textfile.write(element + "\n")

# textfile.close()

# Submission Testset

In [ ]:
!gdown --id 1khuxHPjTTR5d2tL356i8qd-M8JFK2gFu

/usr/local/lib/python3.7/dist-packages/gdown/cli.py:131: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  category=FutureWarning,
Downloading...
From: https://drive.google.com/uc?id=1khuxHPjTTR5d2tL356i8qd-M8JFK2gFu
To: /content/testset_v1_3fnd5.zip
100% 106k/106k [00:00<00:00, 84.2MB/s]


In [ ]:
!unzip -qq /content/testset_v1_3fnd5.zip

In [ ]:
# path_en = '/content/TestSet_ForSuperAI_Release01/test.en'
path_th = '/content/TestSet_ForSuperAI_Release01/test.th'
test_df = pd.DataFrame(extract_data(path_th), columns = ['th_text'])
test_df = test_df[:1520]
test_df

,th_text
0,เขา ทั้ง เตี้ย ทั้ง อ้วน
1,พวก เขา บอก ให้ ฉัน เงียบ
2,คุณ มี แผน อะไร ไหม
3,แสง ไฟ มืด ลง แล้ว
4,คุณ ต้อง ยืนหยัด ดำเนิน การ ตาม แผน
...,...
1515,ตามที่ Henry Akpan หัวหน้านักระบาดวิทยาที่กระท...
1516,ข้อพิพาทในครอบครัวนำไปสู่การหย่าร้างของนาง Moo...
1517,และ Federal Environment Minister ได้รายงานว่า:...
1518,ภาพยนตร์จะนำเสนอภาพเบื้องหลังการซ้อมในช่วงไม่ก...


In [ ]:
to_english = test_df['th_text'].tolist()

In [ ]:
english_preds = model.predict(to_english)

NameError: ignored

In [ ]:
english_preds

["he's both short and fat",
 'they told me to be quiet',
 'do you have any plans',
 'the lights are darkened.',
 'you must insist on the plan.',
 'she needs attention and loving care.',
 'this nurse takes care of the patient with love and patience.',
 "our house doesn 't install air conditioner.",
 'there is milk in this glass.',
 'whose book is this book?',
 'her friend is a liar.',
 'let &apos;s quit school and let &apos;s go to a movie',
 'i have a fortune of 3 baht.',
 'help me with my newspapers.',
 'this winter is colder than last year.',
 'how do other people have any ideas',
 'let &apos;s take those old clothes away',
 "many of our class's classes are very young women boys",
 "i'm moving this box",
 "there's no other way to do it.",
 'she always comes to ask me to help.',
 'is christmas gift finished yet?',
 'the weather of the north is colder.',
 'his nose is big and red.',
 "let's see the traditions of two nations.",
 'i &apos;ve got to go',
 'beijing has changed a lot recent

In [ ]:
def space(preds) :
  lis = []
  for sentence in preds :
    if 'A' <= word[0] <= 'z' and word.lower() in sent_en:
      sent_en.replace(word.lower(),word)
    else :
      pass
    sent_en = sent_en[0].upper()+sent_en[1:]
    lis.append(sent_en)
  return lis

In [ ]:
textfile = open("/content/T5_th2en", "w")

for element in english_preds:

    textfile.write(element + "\n")

textfile.close()

In [ ]:
def post_process(pred):
  post_eng_preds = []
  for sentence in pred:
    post_eng_pred = sentence.replace("&apos;", "'")
    post_eng_preds.append(post_eng_pred)

  return post_eng_preds

In [ ]:
english_preds_new = post_process(english_preds)

In [ ]:
textfile = open("/content/T5_th2en_post", "w")

for element in english_preds_new:

    textfile.write(element + "\n")

textfile.close()